# Reproducing Silkuniene et al. (2025) from dandisets 001047, 001276, 001277 and 001284

Paper: Silkuniene, Silkunas, Pakhomova, Semenov & Pakhomov (2025), "MHz compression of pulse packets facilitates remote focusing of electroporation", Bioelectrochemistry 166, 109016, doi:10.1016/j.bioelechem.2025.109016. The claims file (`claims/silkuniene-2025.yaml`) was written from the accepted author manuscript in PubMed Central (PMC12331426); the paper has no supplement and links no code.

Datasets, each pinned at its most recent published version (the paper cites no version; "Data will be made available on request"):

- DANDI:001047 version 0.250701.1609. Three versions are published; 0.241211.1856 and 0.250701.1609 hold the same 88 assets, 0.240610.2022 holds 54. The latest is pinned; the notebook shows below that none of its files is from the paper's 15.3-mm array, so the choice among versions does not change what can be tested.
- DANDI:001276 version 0.250701.1609, DANDI:001277 version 0.250701.1610, DANDI:001284 version 0.250701.1610: the only published version of each.

Each file is one stitched image of one well. The files are streamed from the archive and reduced to 26-um blocks as they load; reading takes about 20 minutes.

| Claim | Datasets | Status here |
|---|---|---|
| fig1e-field-simulation | all | blocked: data_missing (no field maps) |
| fig2-heating-below-44c | all | blocked: data_missing (no film images) |
| fig3a, fig3bc, fig3-lorentzian, fig3-tail, fig4b (3 claims), fig4c (2 claims) | 001047 | blocked: data_missing (001047 holds no 15.3-mm array images) |
| fig5 (6 claims) | 001284 (+001047) | tested on 001284; fig5c in part |
| fig6 (5 claims) | 001277 | tested |
| fig7 (6 claims) | 001276 (+ others for fig7-sharpest) | tested |

In [ ]:
import os
os.environ.setdefault("OMP_NUM_THREADS", "1")
import time, resource
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats
import cancan as cc

pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 40)
pd.set_option("display.max_rows", 400)
T0 = time.time()
print("pinned versions:", cc.VERSIONS)

## Data check

In [ ]:
meta = pd.DataFrame(cc.metadata_table())
print(len(meta), "files")
print(meta.groupby(["dandiset", "cell", "channel"], dropna=False).size().to_string())
print("inter-electrode distance stated in the files (mm):")
print(meta.groupby(["dandiset", "spacing_mm", "first_pulse_kv"], dropna=False).size().to_string())
print("pixel size (um) in the OME metadata:", sorted(meta.pixel_um.dropna().unique()))
meta["extent_mm"] = [tuple(round(s * p / 1000, 1) for s in sh[-2:]) for sh, p in zip(meta["shape"], meta.pixel_um.fillna(cc.PIXEL_UM))]
print(meta.groupby(["dandiset", "extent_mm"]).size().to_string())

DANDI:001047. Every file states a 3.8-mm inter-electrode distance and a 3.2-kV first pulse; the paper's array has 15.3-mm diagonals between 2.8-mm needles (12.8 mm between needle centres along a side) and a 7.2-kV first pulse (Methods 2.2). The CHO files cover 13.0 x 13.8 mm. Two of them are shown, reduced, to check the stated geometry against the images.

In [ ]:
m47 = meta[meta.dandiset == "001047"]
print(m47.groupby(["subject", "channel", "protocol", "packet_rate", "repeats"], dropna=False).size().to_string())
ex47 = [m47[(m47.cell == "CHO") & (m47.channel == "FITC")].iloc[0], m47[(m47.channel == "FITC") & (m47.extent_mm.map(lambda e: e[1] > 25))].iloc[0]]
fig, ax = plt.subplots(1, 2, figsize=(14, 5))
for a, r in zip(ax, ex47):
    im = cc.read_reduced(r.asset_id)[0]
    v = im[im > 0]
    a.imshow(np.clip(im, *np.percentile(v, [1, 99.5])), cmap="gray",
             extent=[0, im.shape[1] * cc.MM_PER_PX, im.shape[0] * cc.MM_PER_PX, 0])
    a.set_title(f"001047 {r.subject} {r.channel}, {r.spacing_mm} mm stated"); a.set_xlabel("mm"); a.set_ylabel("mm")
    print(r.path, "extent (mm):", r.extent_mm)
plt.tight_layout(); plt.show()

The images show electrode footprints a few mm apart, two exposure areas side by side in the wide images, as the files describe: these are experiments with a 3.8-mm array, not the paper's 15.3-mm array. The paper's packet-rate series of Figures 3 and 4 (1 Hz to 0.2 MHz on the 15.3-mm array) is not in DANDI:001047.

The other three dandisets hold the 15.3-mm array. The files used below: YO-PRO-1 (FITC) images of 001276, 001277 and 001284, and the propidium and Hoechst images taken 2 h after exposure (two-channel Cy3-DAPI files) of 001284.

In [ ]:
use = meta[((meta.dandiset.isin(["001276", "001277", "001284"])) & (meta.channel == "FITC"))
           | ((meta.dandiset == "001284") & (meta.channel == "Cy3-DAPI"))].copy()
print(len(use), "files to read")
t = time.time()
imgs = cc.read_many(list(use.asset_id), workers=4)
images = dict(zip(use.asset_id, imgs))
print(f"read in {time.time() - t:.0f} s; peak memory {resource.getrusage(resource.RUSAGE_SELF).ru_maxrss / 1e6:.2f} GB")
use["frac_empty"] = [float((images[a][0] == 0).mean()) for a in use.asset_id]
empty = use[use.frac_empty > 0.9]
print("files that are more than 90% empty (zero) pixels, left out:")
print(empty[["dandiset", "path", "channel", "size", "frac_empty"]].to_string())
use = use[use.frac_empty <= 0.9]

In [ ]:
t = time.time()
W = cc.well_profiles(use.to_dict("records"), images, workers=4)
print(f"profiles in {time.time() - t:.0f} s")

def summary_row(o):
    h = o["pa"]["half"]
    return dict(dandiset=o["dandiset"], well=o["subject"], chan=o["chan"], cell=o["cell"],
                protocol=o["protocol"], rate=o["packet_rate"], bursts=o["repeats"], pulse_ns=o["pulse_ns"],
                elev_mm=o["elevation_mm"], sham=o["sham"], geometry=o["how"], n_squares_found=o["n_found"],
                side_mm=o["geom"]["side_mm"], theta=o["geom"]["theta_deg"], bg=o["pa"]["background"],
                c0=h[:, 0].mean(), y6=h[:, 6].mean(), y7=h[:, 7].mean())
S = pd.DataFrame([summary_row(o) for o in W])

Locating the array. The paper measures from the geometrical centre of the four electrodes (Methods 2.5). The footprints are found in each image as dark disks or bright rings of the needle's 2.8-mm diameter at the corners of a square (`cc.find_array`). A square is accepted when each corner scores at least 0.15 and its side is within 0.6 mm of 12.8 mm (standard choices, from inspecting the detections). In many images the footprints are faint (the shading of the stitched tiles is stronger) and with the array raised 1 or 2 mm there are none. For those wells the centre is the centre of the central YP spot, with the square's rotation set to the median of the squares found; this is an assumption, checked below on the wells where both are available.

In [ ]:
F = S[S.chan == "FITC"]
print(F.groupby(["dandiset", "geometry"]).size().to_string())
ok = F[F.geometry == "footprints"]
print(f"squares found: side {ok.side_mm.min():.2f} to {ok.side_mm.max():.2f} mm (median {ok.side_mm.median():.2f}; nominal {cc.SIDE_MM:.2f}); rotation {ok.theta.min():.1f} to {ok.theta.max():.1f} deg (median {ok.theta.median():.1f})")
# where footprints were found, how far is the YP spot centre from the geometric centre?
off = []
for o in W:
    if o["chan"] == "FITC" and o["how"] == "footprints" and o["protocol"] == "CanCan":
        sc = cc.spot_centre(images[o["asset_id"]][0])
        off.append(dict(well=o["subject"], dandiset=o["dandiset"], rate=o["packet_rate"], dx=(sc[0] - o["geom"]["centre"][0]) * cc.MM_PER_PX, dy=(sc[1] - o["geom"]["centre"][1]) * cc.MM_PER_PX))
off = pd.DataFrame(off); off["dist"] = np.hypot(off.dx, off.dy)
print("CANCAN wells with footprints: distance of the YP spot centre from the array centre (mm)")
print(off.groupby("rate").dist.describe().round(2).to_string())
print("wells with the spot more than 1.5 mm off centre:"); print(off[off.dist > 1.5].round(2).to_string())
# 001284: do the FITC and the 2-h images of one well share a frame? Distance between the
# YP spot centre and the propidium spot centre of the same well (0-mm wells, where both spots are strong)
al = []
for o in W:
    if o["dandiset"] == "001284" and o["chan"] == "Cy3" and o["elevation_mm"] == "0":
        fitc = [q for q in W if q["subject"] == o["subject"] and q["chan"] == "FITC"][0]
        a_ = cc.spot_centre(images[fitc["asset_id"]][0]); b_ = cc.spot_centre(images[o["asset_id"]][0])
        al.append(dict(well=o["subject"], shift_mm=float(np.hypot(*np.subtract(a_, b_)) * cc.MM_PER_PX)))
print("YP spot centre against propidium spot centre, same well, 0 mm:", pd.DataFrame(al).round(2).to_dict("records"))

In [ ]:
fig, axs = plt.subplots(8, 16, figsize=(32, 17))
for a in axs.ravel(): a.axis("off")
for a, o in zip(axs.ravel(), [o for o in W if o["chan"] in ("FITC", "Cy3")]):
    im = images[o["asset_id"]][0]; v = im[im > 0]
    a.imshow(np.clip(im, *np.percentile(v, [1, 99.7])), cmap="gray")
    c = o["geom"]["corners"]; a.plot(*np.vstack([c, c[:1]]).T, "r-", lw=0.6); a.plot(*o["geom"]["centre"], "r+")
    a.set_title(f"{o['dandiset'][-3:]} {o['subject'][:2]}{o['subject'][-9:]} {o['chan']} {o['how'][:4]}", fontsize=6)
plt.tight_layout(); plt.show()

In [ ]:
print("wells by dandiset, cell line, protocol, packet rate, bursts (x4), pulse duration and elevation (FITC images):")
print(S[S.chan == "FITC"].groupby(["dandiset", "cell", "protocol", "rate", "bursts", "pulse_ns", "elev_mm", "sham"], dropna=False).size().to_string())
print("\npropidium (Cy3) wells of 001284 by elevation:", S[S.chan == "Cy3"].groupby("elev_mm").size().to_dict())
print("\nbackground (mean of 3 ROIs in the darkest footprint), raw a.u.:")
print(S.groupby(["dandiset", "chan"]).bg.describe().round(0).to_string())
print("\ncentre value after background subtraction, a.u.:")
print(S.groupby(["dandiset", "chan"]).c0.describe().round(0).to_string())
print("\nimage values are uint16 counts, unit 'n.a.' in the files; pixel", cc.PIXEL_UM, "um; blocks of", cc.BLOCK, "pixels =", round(cc.MM_PER_PX * 1000), "um")
print("ROI positions, reading (a):", cc.DIST_A, "mm; reading (b):", np.round(cc.DIST_B, 2), "mm")
conf = meta[meta.dandiset.isin(["001277", "001284"])].groupby(["dandiset", "subject"]).pulse_ns.agg(lambda v: sorted(set(v.dropna())))
print("\nwells whose files state more than one pulse duration:", conf[conf.map(len) > 1].to_dict())
print("packets per burst stated in the 001277 files:", sorted(set(meta[meta.dandiset == "001277"].packet_rate.dropna())), "(every file: '9 packets of pulses')")
dup = meta.groupby(["dandiset", "size"]).filter(lambda g: len(g) > 1)
print("\nfiles of identical size (checked for duplicates below):"); print(dup[["dandiset", "path", "channel", "size"]].to_string())

Duplicate check: the two Cy3-DAPI files of 001277 well P2-20240925-B3 have the same size. A block of each is compared.

In [ ]:
for _, g in dup.groupby(["dandiset", "size"]):
    a, b = g.asset_id.tolist()[:2]
    fa, da = cc._open(a); fb, db = cc._open(b)
    sl = (0, slice(None), slice(9000, 9300), slice(9000, 9300)) if da["data"].ndim == 4 else (0, slice(9000, 9300), slice(9000, 9300))
    same = np.array_equal(da["data"][sl], db["data"][sl])
    print(g.path.tolist(), "identical block:", same)
    fa.close(); fb.close()

Checks of the description against the data, stated here and taken up in the claims below:

- The pixel size (1.625 um) and the needle diameter agree: the footprints found have the 2.8-mm diameter, and the squares found have sides near the 12.8 mm the paper's geometry gives (printed above). The dandisets' descriptions give a 10-mm inter-electrode distance, which matches no distance of this array (edge to edge along a side is 10.0 mm only if measured between needle centres minus the needle diameter; this is consistent with the data, not shown by it).
- 001277: every file says "9 packets ... at 0.2MHz" whatever the pulse duration, where the dandiset description and the paper give 18, 9 and 6 packets. One well (P1-20240926-B1) is a sham exposure. The 600-ns group holds 7 wells, where the paper reports n = 5; two of them (P1-20240926-B2 and B3) show broad uptake with no central peak (montage above, and below).
- 001276: besides the 0.2-MHz burst series it holds CANCAN wells at 1 Hz and control wells, which its description does not mention.
- 001284: the YP groups hold 12, 11 and 11 wells at 0, 1 and 2 mm (paper: n = 6 to 7); the propidium groups hold 5, 5 and 4 (paper: 4 to 5).
- There is no time base to check: each file is a single image. Images of one well taken at different times share one stage frame where this could be checked (001284, above).

## Common definitions

Each experiment (well) gives a profile from its ROIs. ROI positions (claims file O1): reading (a) at whole mm, reading (b) contiguous 0.406-mm steps. Pooling of the diagonals (O2): reading (a) the two diagonals averaged at each signed distance, (b) the four half-diagonals averaged and mirrored. A value "at d mm" is the mean of the four half-diagonals at the ROI nearest d. The paper's graph rule (F8) is `cc.gap_exceeds`. Student's t-tests are two-sided and unpaired (F7).

In [ ]:
def prof(o, roi="a", pool="a"):
    return cc.pooled(o["pa"] if roi == "a" else o["pb"], pool)

def at(o, d, roi="a"):
    p = o["pa"] if roi == "a" else o["pb"]
    i = int(np.argmin(np.abs(p["dists"] - d)))
    return float(p["half"][:, i].mean())

def ms(v):
    v = np.asarray(v, float); v = v[np.isfinite(v)]
    return v.mean(), (v.std(ddof=1) / np.sqrt(len(v)) if len(v) > 1 else np.nan), len(v)

def fwhm_each(group, roi, half_range, func="lorentz"):
    out = []
    for o in group:
        x, y = prof(o, roi, "a")
        out.append(cc.fit(x, y, func, half_range)["fwhm"])
    return np.array(out)

def mean_profile(group, roi="a", pool="a", norm=None):
    ys = []
    for o in group:
        x, y = prof(o, roi, pool)
        ys.append(y)
    ys = np.array(ys)
    return x, ys

def wells(dandiset, chan="FITC", **kw):
    out = [o for o in W if o["dandiset"] == dandiset and o["chan"] == chan]
    for k, v in kw.items():
        out = [o for o in out if o[k] == v]
    return out

ROIS, POOLS, RANGES = ("a", "b"), ("a", "b"), (4.0, 5.0, 6.0)

## Claims not tested

- fig1e-field-simulation: the simulated field maps (Sim4Life) are in none of the four dandisets; blocked, data_missing.
- fig2-heating-below-44c: the thermochromic film images are in none of the four dandisets; blocked, data_missing.
- fig3a-centre-enhanced, fig3bc-rate-suppresses-off-target, fig3-lorentzian-profile, fig3-tail-deviation-falls-with-rate, fig4b-40-fold-ratio, fig4b-centre-no-rate-effect, fig4b-near-electrode-decline, fig4c-fwhm-falls-with-rate, fig4c-log-trend: the claims name DANDI:001047, which holds only 3.8-mm array images (above); blocked, data_missing. DANDI:001276 holds CT26 wells at 1 Hz (n = 2), controls (n = 4) and 0.2 MHz (n = 6) at 2 bursts (x4), but no 1-kHz or 10-kHz wells and no parallel ROIs; those wells are shown here only as a variant, for fig4b-40-fold-ratio, without deciding any outcome.

In [ ]:
ct2 = wells("001276", cell="CT26", repeats="2")
print("001276 CT26, 2 bursts (x4), by protocol and rate:")
v = pd.DataFrame([dict(well=o["subject"], protocol=o["protocol"], rate=o["packet_rate"], c0=at(o, 0), y6=at(o, 6), y7=at(o, 7)) for o in ct2])
v["c0/y6"] = v.c0 / v.y6; v["c0/y7"] = v.c0 / v.y7
print(v.round(1).to_string())
g = v[(v.protocol == "CanCan") & (v.rate == "0.2MHz")]
print(f"variant (not deciding fig4b-40-fold-ratio), 0.2 MHz n={len(g)}: ratio of means {g.c0.mean() / g.y6.mean():.1f} (6 mm), {g.c0.mean() / g.y7.mean():.1f} (7 mm); mean of ratios {g['c0/y6'].mean():.1f}, {g['c0/y7'].mean():.1f}")

## Figure 5: raised electrodes and cell killing (DANDI:001284)

### fig5-raised-electrodes-weaken-centre

> Since with MHz-CANCAN there was practically no YP staining near electrodes (a finding the text cites to Figure 4), the only clear effect of raising the electrodes 1 or 2 mm above the monolayer was a weakening of electroporation in the central spot.

Centre YP in % of the mean 0-mm centre (Fig. 5B legend); 1 and 2 mm against 0 mm by the graph rule (F8), with Student's t-tests, Bonferroni over the two. The centre ROI is the same under every reading of O1 and O2.

In [ ]:
yp = {e: wells("001284", elevation_mm=e) for e in ("0", "1", "2")}
c0ref = np.mean([at(o, 0) for o in yp["0"]])
rows = []
for e, g in yp.items():
    c = np.array([at(o, 0) for o in g]) / c0ref * 100
    m, se, n = ms(c)
    rows.append(dict(elev=e, n=n, centre_pct=m, se=se, y6_pct=ms([at(o, 6) / c0ref * 100 for o in g])[0], y7_pct=ms([at(o, 7) / c0ref * 100 for o in g])[0]))
R5 = pd.DataFrame(rows); print(R5.round(2).to_string())
c = {e: np.array([at(o, 0) for o in g]) / c0ref * 100 for e, g in yp.items()}
for e in ("1", "2"):
    t = stats.ttest_ind(c[e], c["0"])
    print(f"{e} mm vs 0 mm: gap rule {cc.gap_exceeds(c['0'].mean(), ms(c['0'])[1], c[e].mean(), ms(c[e])[1])}; t = {t.statistic:.2f}, p = {t.pvalue:.4f}, Bonferroni x2 p = {min(1, 2 * t.pvalue):.4f}")
# by batch of experiments (dates), as the dandiset holds more wells than the paper reports
for batch, sel in (("Aug 2024", lambda o: "202408" in o["subject"]), ("Oct 2024", lambda o: "202410" in o["subject"])):
    print(batch, {e: (round(np.mean([at(o, 0) / c0ref * 100 for o in g if sel(o)]), 1), sum(sel(o) for o in g)) for e, g in yp.items()})

**Result: partially reproduced.** Raising the electrodes lowered mean centre YP to 48.7% (1 mm, n = 11) and 28.9% (2 mm, n = 11) of the 0-mm value (n = 12). Only the 2-mm drop is larger than the error bars by the paper's graph rule; the 1-mm drop is not (t-test p = 0.079, Bonferroni 0.158; 2 mm p = 0.028, Bonferroni 0.056). YP at 6 and 7 mm is 1.5 to 7.4% of the 0-mm centre at every elevation. In the October 2024 wells alone the centre falls from 121.8% to 38.0% and 7.1%.

### fig5-killing-focused-centre

> With MHz-CANCAN, cell killing (propidium uptake 2 h after exposure) was produced at the centre of the array with negligible effect near the electrodes.

Pr at x = 0 against 6 and 7 mm at 0 mm elevation, graph rule and Student's t-tests; a paired t-test (same wells) as a variant.

In [ ]:
pr = {e: wells("001284", "Cy3", elevation_mm=e) for e in ("0", "1", "2")}
for e, g in pr.items():
    for roi in ROIS:
        c0 = np.array([at(o, 0, roi) for o in g]); y6 = np.array([at(o, 6, roi) for o in g]); y7 = np.array([at(o, 7, roi) for o in g])
        m0, s0, n = ms(c0)
        line = f"elev {e} mm, ROI reading {roi}, n={n}: centre {m0:.0f} +/- {s0:.0f}"
        for d, y in ((6, y6), (7, y7)):
            m, s, _ = ms(y)
            t = stats.ttest_ind(c0, y); tp = stats.ttest_rel(c0, y)
            line += f"; {d} mm {m:.1f} +/- {s:.1f} ({100 * m / m0:.1f}% of centre), gap {cc.gap_exceeds(m0, s0, m, s)}, t p={t.pvalue:.4f}, paired p={tp.pvalue:.4f}"
        print(line)

**Result: reproduced.** At 0 mm (n = 5), centre Pr is 1188 +/- 339 a.u. and Pr at 6 and 7 mm is 8.2 to 16.5 a.u. (0.7 to 1.4% of the centre), below the centre by more than the error bars under both ROI readings (Student's t-test p = 0.008 to 0.009; paired p = 0.025). At 1 mm Pr at 6 and 7 mm is about 13% of the centre and the gap holds; at 2 mm the centre Pr is 23 a.u. and it does not.

### fig5ab-yp-spot-slightly-larger-than-pr

> The central spot of YP staining (electroporation) was only slightly larger than the spot of Pr staining (cell killing).

FWHM of the Lorentzian fit to the mean YP profile and of the Gaussian fit to the mean Pr profile, -5 to 5 mm, in % of the 0-mm peak (Fig. 5B legend); Gaussian FWHM 2.4 sigma (Methods) and 2.355 sigma (variant). Ratio judged against 1 to 1.25 (reading a) and 1 to 1.5 (reading b). The YP wells of 0 mm are all 12 (main), and the 5 wells that also have a Pr image as a variant.

In [ ]:
def pct_profile(group, roi, pool, ref):
    x, ys = mean_profile(group, roi, pool)
    return x, ys / ref * 100

rows = []
for e in ("0", "1", "2"):
    for roi in ROIS:
        for pool in POOLS:
            for ypset, ypg in (("all YP wells", yp[e]), ("wells with Pr", [o for o in yp[e] if o["subject"] in {p["subject"] for p in pr[e]}])):
                xr, ysy = pct_profile(ypg, roi, pool, np.mean([at(o, 0, roi) for o in yp["0"]]))
                xp, ysp = pct_profile(pr[e], roi, pool, np.mean([at(o, 0, roi) for o in pr["0"]]))
                fy = cc.fit(xr, ysy.mean(0), "lorentz", 5.0); fp = cc.fit(xp, ysp.mean(0), "gauss", 5.0)
                rows.append(dict(elev=e, roi=roi, pool=pool, yp_set=ypset, n_yp=len(ypg), n_pr=len(pr[e]), fwhm_yp=fy["fwhm"], fwhm_pr=fp["fwhm"], ratio=fy["fwhm"] / fp["fwhm"], ratio_2355=fy["fwhm"] / (fp["w"] * 2.355)))
R5ab = pd.DataFrame(rows); print(R5ab.round(3).to_string())
z = R5ab[(R5ab.elev == "0") & (R5ab.yp_set == "all YP wells")]
print(f"0 mm, all YP wells: ratio {z.ratio.min():.3f} to {z.ratio.max():.3f}; within 1-1.25 under every reading: {bool(((z.ratio >= 1) & (z.ratio <= 1.25)).all())}; within 1-1.5: {bool(((z.ratio >= 1) & (z.ratio <= 1.5)).all())}")
z2 = R5ab[(R5ab.elev == "0") & (R5ab.yp_set == "wells with Pr")]
print(f"0 mm, wells with Pr: ratio {z2.ratio.min():.3f} to {z2.ratio.max():.3f}")

**Result: not reproduced.** At 0 mm the FWHM of the Lorentzian fit to mean YP is 1.69 to 1.75 mm and that of the Gaussian fit to mean Pr 2.29 to 2.34 mm, a ratio of 0.74 to 0.75 under every reading (0.66 to 0.67 with only the 5 wells that have a Pr image; 0.75 to 0.76 with FWHM = 2.355 sigma). By FWHM the YP spot is smaller than the Pr spot, outside 1 to 1.25 and 1 to 1.5.

### fig5b-gaussian-pr-lorentzian-yp

> Pr intensity declined from the peak as a Gaussian and YP as a Lorentzian; in the left panel of Figure 5B the R2 was 0.92 (Lorentzian) against 0.96 (Gaussian) for Pr, and 0.98 against 0.95 for YP, fitted from -5 to 5 mm with p < 0.0001.

R2 = 1 - SSres/SStot of each fit to the mean 0-mm profile in %, -5 to 5 mm; the F test of the fit is printed.

In [ ]:
rows = []
for roi in ROIS:
    for pool in POOLS:
        xr, ysy = pct_profile(yp["0"], roi, pool, np.mean([at(o, 0, roi) for o in yp["0"]]))
        xp, ysp = pct_profile(pr["0"], roi, pool, np.mean([at(o, 0, roi) for o in pr["0"]]))
        r = dict(roi=roi, pool=pool)
        for name, (x, y) in (("pr", (xp, ysp.mean(0))), ("yp", (xr, ysy.mean(0)))):
            for func in ("lorentz", "gauss"):
                f = cc.fit(x, y, func, 5.0)
                r[f"{name}_{func}_r2"] = f["r2"]; r[f"{name}_{func}_p"] = f["p_f"]; r[f"{name}_n"] = f["n"]
        rows.append(r)
R5b = pd.DataFrame(rows); print(R5b.to_string(float_format=lambda v: f"{v:.3g}"))
exp = dict(pr_lorentz_r2=0.92, pr_gauss_r2=0.96, yp_lorentz_r2=0.98, yp_gauss_r2=0.95)
for k, v in exp.items():
    print(f"{k}: expected {v} +/- {0.1 * v:.3f}; measured {R5b[k].min():.3f} to {R5b[k].max():.3f}; within tolerance under every reading: {bool((abs(R5b[k] - v) <= 0.1 * v).all())}")
print("Pr: Gaussian above Lorentzian under every reading:", bool((R5b.pr_gauss_r2 > R5b.pr_lorentz_r2).all()))
print("YP: Lorentzian above Gaussian under every reading:", bool((R5b.yp_lorentz_r2 > R5b.yp_gauss_r2).all()))

**Result: reproduced.** R2 of the Lorentzian and Gaussian fits are 0.937 to 0.960 and 0.984 to 0.998 for Pr, and 0.975 to 0.993 and 0.960 to 0.979 for YP, each within 10% of 0.92, 0.96, 0.98 and 0.95, with the Gaussian better for Pr and the Lorentzian better for YP under every reading. Every fit's F test has p below 0.0001.

### fig5b-reversible-beyond-2mm

> The Lorentzian YP profile and the Gaussian Pr profile differ mainly in the tails, where YP is broader, marking an area of predominantly reversible electroporation that starts about 2 mm from the centre.

YP and Pr each in % of its own 0-mm peak. Reading (a): the smallest distance at which mean YP% exceeds mean Pr% by a gap larger than the error bars; reading (b): the smallest distance at which the fitted Lorentzian (in % of its a) exceeds the fitted Gaussian (in % of its a) by more than the mean s.e. of YP%. Each side separately (reading a of O2), then averaged.

In [ ]:
rows = []
for roi in ROIS:
    xr, ysy = pct_profile(yp["0"], roi, "a", np.mean([at(o, 0, roi) for o in yp["0"]]))
    xp, ysp = pct_profile(pr["0"], roi, "a", np.mean([at(o, 0, roi) for o in pr["0"]]))
    my, sy = ysy.mean(0), ysy.std(0, ddof=1) / np.sqrt(len(ysy))
    mp, sp = ysp.mean(0), ysp.std(0, ddof=1) / np.sqrt(len(ysp))
    fy = cc.fit(xr, my, "lorentz", 5.0); fp = cc.fit(xp, mp, "gauss", 5.0)
    mean_se = np.mean(sy[np.abs(xr) <= 5])
    xs = np.arange(0, 7.001, 0.01)
    dfit = 100 * cc.lorentz(xs, 1, fy["w"]) - 100 * cc.gauss(xs, 1, fp["w"])
    d_b = xs[np.argmax(dfit > mean_se)] if (dfit > mean_se).any() else np.nan
    sides = []
    for sign in (-1, 1):
        idx = [i for i in np.argsort(np.abs(xr)) if np.sign(xr[i]) == sign]
        hit = [abs(xr[i]) for i in idx if cc.gap_exceeds(my[i], sy[i], mp[i], sp[i]) and my[i] > mp[i]]
        sides.append(min(hit) if hit else np.nan)
    rows.append(dict(roi=roi, side_neg=sides[0], side_pos=sides[1], reading_a=np.nanmean(sides), reading_b=d_b, mean_se_yp=mean_se))
    print(f"ROI reading {roi}: x = {np.round(xr[xr >= 0], 2).tolist()}")
    print("   YP% ", np.round(my[xr >= 0], 1).tolist(), " s.e.", np.round(sy[xr >= 0], 1).tolist())
    print("   Pr% ", np.round(mp[xr >= 0], 1).tolist(), " s.e.", np.round(sp[xr >= 0], 1).tolist())
R5r = pd.DataFrame(rows); print(R5r.round(3).to_string())

**Result: reproduced.** Under reading (b), the fitted Lorentzian YP profile exceeds the fitted Gaussian Pr profile by more than the mean s.e. of YP% from 2.08 mm (whole-mm ROIs) and 2.11 mm (contiguous ROIs), inside 1.8 to 2.2 mm. Under reading (a), on the data by the graph rule, the first such distance is 3.0 mm (whole-mm ROIs) and 2.84 mm (contiguous ROIs, the mean of 3.25 and 2.44 mm on the two sides), outside the range; the criterion asks for one reading.

### fig5c-nuclear-yp-centre

> In the central spot cells showed bright nuclear YP staining, indicating irreversible damage, while farther out reversibly electroporated cells were weakly stained with less contrast between nucleus and cytoplasm; cells inside the electrode footprint were unstained though present (Hoechst).

The high-resolution images of Fig. 5C are not in the dandiset (every 001284 file is a 4x whole-well image), so the nuclear part cannot be tested. Intensity part on the 0-mm whole-well images: background-subtracted YP in zone 1 (within 1 mm) against zone 2 (2 to 4 mm, reading a; 3 to 5 mm, reading b) and zone 3 (the 0.5-mm band outside a wall, 7.15 to 7.65 mm), and footprint YP (mean of the four footprints) below zones 2 and 3. Footprint part: the Hoechst channel of the 2-h images, footprint mean against the camera offset (0.1st percentile of the image, a standard choice) and against the well's median; cells are taken as present when the footprint is at least half-way from the offset to the median (a standard choice).

In [ ]:
rows = []
for o in yp["0"]:
    z = o["zone"]; b = z["background"]
    rows.append(dict(well=o["subject"], zone1=z["zone1"] - b, zone2a=z["zone2a"] - b, zone2b=z["zone2b"] - b, zone3=z["zone3"] - b, footprint=z["footprint"] - b))
Z = pd.DataFrame(rows); print(Z.round(1).to_string())
m = Z.mean(numeric_only=True); s = Z.sem(numeric_only=True)
print("means:", m.round(1).to_dict())
for z2 in ("zone2a", "zone2b"):
    print(f"zone1 > {z2}: {bool(m.zone1 > m[z2])} (gap rule {cc.gap_exceeds(m.zone1, s.zone1, m[z2], s[z2])}); zone1 > zone3: {bool(m.zone1 > m.zone3)}; footprint < {z2}: {bool(m.footprint < m[z2])}; footprint < zone3: {bool(m.footprint < m.zone3)}")
print("wells where every inequality holds (zone2a):", int(((Z.zone1 > Z.zone2a) & (Z.zone1 > Z.zone3) & (Z.footprint < Z.zone2a) & (Z.footprint < Z.zone3)).sum()), "of", len(Z))
rows = []
for o in wells("001284", "DAPI_2h", elevation_mm="0"):
    z = o["zone"]
    rows.append(dict(well=o["subject"], footprint=z["footprint"], offset=z["offset"], median=z["median"], frac=(z["footprint"] - z["offset"]) / (z["median"] - z["offset"])))
H = pd.DataFrame(rows); print("Hoechst 2 h after exposure, 0 mm:"); print(H.round(2).to_string())
print("wells with footprint Hoechst at least half-way from offset to median:", int((H.frac >= 0.5).sum()), "of", len(H))

**Result: partially reproduced.** Intensity part: mean background-subtracted YP is 1607.5 a.u. in zone 1, 140.2 (2 to 4 mm) or 86.1 (3 to 5 mm) in zone 2, 320.8 in zone 3 and 10.5 in the footprints; every inequality holds in 12 of 12 wells. Footprint part: in the 5 wells with a Hoechst image taken 2 h after exposure, the footprint Hoechst signal is above the well's median (1.21 to 1.57 of the way from the camera offset to the median), so cells are present. The nuclear part cannot be tested, as the high-resolution images of Fig. 5C are not in the dandiset.

## Figure 6: pulse duration (DANDI:001277)

The paper reports n = 5 per duration. The dandiset's 600-ns group holds 7 wells; two (P1-20240926-B2 and B3) have no central peak (montage above; FWHM printed below). The main reading uses the five other 600-ns wells, so that n = 5 as the claims state; all seven are shown as a variant. The sham well is left out.

In [ ]:
f6 = {d: wells("001277", pulse_ns=d, sham=False) for d in ("300", "600", "900")}
odd = {"P1-20240926-B2", "P1-20240926-B3"}
for d, g in f6.items():
    print(d, [(o["subject"], round(at(o, 0)), round(cc.fit(*prof(o), "lorentz", 4.0)["fwhm"], 2)) for o in g])
f6_main = {d: [o for o in g if o["subject"] not in odd] for d, g in f6.items()}
print("n per duration, main:", {d: len(g) for d, g in f6_main.items()}, "variant:", {d: len(g) for d, g in f6.items()})

### fig6b-peak-900ns-not-significant

> The central YP peak was highest for 900-ns pulses, but not significantly different from 300- or 600-ns pulses.

In [ ]:
for label, groups in (("main (n = 5 each)", f6_main), ("variant, all 600-ns wells", f6)):
    c = {d: np.array([at(o, 0) for o in g]) for d, g in groups.items()}
    print(label, {d: (round(v.mean()), round(ms(v)[1])) for d, v in c.items()}, "highest:", max(c, key=lambda d: c[d].mean()))
    for other in ("300", "600"):
        t = stats.ttest_ind(c["900"], c[other])
        dz = (c["900"].mean() - c[other].mean()) / np.sqrt((c["900"].var(ddof=1) + c[other].var(ddof=1)) / 2)
        print(f"   900 vs {other}: t = {t.statistic:.2f}, p = {t.pvalue:.3f}, Bonferroni x2 p = {min(1, 2 * t.pvalue):.3f}, x3 p = {min(1, 3 * t.pvalue):.3f}; Cohen's d = {dz:.2f}")
    print(f"   one-way ANOVA p = {stats.f_oneway(*c.values()).pvalue:.3f}; 300 vs 600 p = {stats.ttest_ind(c['300'], c['600']).pvalue:.3f}")

**Result: reproduced.** Mean centre YP is 538, 800 and 891 a.u. for 300, 600 and 900 ns (n = 5 each), highest at 900 ns; 900 against 300 ns p = 0.288 (Bonferroni 0.576), against 600 ns p = 0.801 (Bonferroni 1.0). With all seven 600-ns wells 900 ns is still highest and no test is significant (ANOVA p = 0.51 and 0.47).

### fig6c-normalized-decline-similar

> Normalized to the central peak, the sideways decline of YP was similar for the three pulse durations.

Each experiment in % of its value at x = 0, then mean +/- s.e. per duration; at every distance every pair of durations is checked with the graph rule. Variants: normalising the mean profile, and -5 to 5 mm only.

In [ ]:
for label, groups in (("main", f6_main), ("variant, all 600-ns wells", f6)):
    for roi in ROIS:
        for pool in POOLS:
            prof_pct = {}
            for d, g in groups.items():
                x, ys = mean_profile(g, roi, pool)
                prof_pct[d] = (x, ys / ys[:, [np.argmin(np.abs(x))]] * 100)
            hits = []
            for i, xi in enumerate(x):
                if xi == 0: continue
                for a_, b_ in (("300", "600"), ("300", "900"), ("600", "900")):
                    ya, yb = prof_pct[a_][1][:, i], prof_pct[b_][1][:, i]
                    if cc.gap_exceeds(ya.mean(), ms(ya)[1], yb.mean(), ms(yb)[1]):
                        hits.append((round(float(xi), 2), a_, b_, round(ya.mean(), 1), round(yb.mean(), 1)))
            within5 = [h for h in hits if abs(h[0]) <= 5]
            print(f"{label}, ROI {roi}, pooling {pool}: {len(hits)} distance-pairs with a gap ({len(within5)} within 5 mm): {hits}")
x, _ = mean_profile(f6_main["300"])
for d, g in f6_main.items():
    xx, ys = mean_profile(g); mp = ys.mean(0); print(d, "mean profile normalised to its peak (%):", np.round(mp / mp[np.argmin(np.abs(xx))] * 100, 1)[xx >= 0].tolist())

**Result: not reproduced.** By the paper's graph rule, durations differ by a gap larger than the error bars at some distances under every reading. With n = 5 per duration there are 5 to 20 such distance and pair combinations, nearly all within 5 mm, mostly 300 ns above 600 or 900 ns at 1 to 3 mm (whole-mm ROIs at 2 mm: 20.2% against 12.3% of the peak for 300 and 600 ns). With all seven 600-ns wells there are 2 to 10, all 300 against 900 ns. The 300-ns peak is broader, which is what fig6d-longer-pulses-sharper-peak measures.

### fig6d-longer-pulses-sharper-peak

> Fitted per experiment, the FWHM of the central YP peak decreases with pulse duration from 300 to 900 ns, with a trend slope different from zero (p = 0.003).

Per-experiment Lorentzian FWHM (a and b free, centre fixed, no offset), over the fit ranges of O3 (4, 5, 6 mm) and the ROI readings of O1; O2 does not change a fit to a symmetric function of x. Slope of FWHM against duration and against log duration, on individual experiments (O4 a) and on group means (O4 b).

In [ ]:
rows = []
for label, groups in (("main", f6_main), ("all 600", f6)):
    for roi in ROIS:
        for hr in RANGES:
            fw = {d: fwhm_each(g, roi, hr) for d, g in groups.items()}
            X = np.concatenate([[int(d)] * len(v) for d, v in fw.items()]); Y = np.concatenate(list(fw.values()))
            Xm = np.array([int(d) for d in fw]); Ym = np.array([v.mean() for v in fw.values()])
            r = dict(set=label, roi=roi, range=hr, **{f"fwhm_{d}": v.mean() for d, v in fw.items()}, **{f"se_{d}": ms(v)[1] for d, v in fw.items()})
            for nm, (xx, yy) in (("lin_exp", (X, Y)), ("log_exp", (np.log(X), Y)), ("lin_means", (Xm, Ym)), ("log_means", (np.log(Xm), Ym))):
                st = cc.slope_test(xx, yy); r[f"{nm}_slope"] = st["slope"]; r[f"{nm}_p"] = st["p"]
            rows.append(r)
R6d = pd.DataFrame(rows); print(R6d.round(4).to_string())
mm = R6d[R6d.set == "main"]
print("main: every slope negative:", bool((mm[[c for c in mm if c.endswith('_slope')]] < 0).all().all()),
      "; p on experiments", f"{mm[['lin_exp_p', 'log_exp_p']].min().min():.4f} to {mm[['lin_exp_p', 'log_exp_p']].max().max():.4f}",
      "; p on means", f"{mm[['lin_means_p', 'log_means_p']].min().min():.3f} to {mm[['lin_means_p', 'log_means_p']].max().max():.3f}")

**Result: reproduced, resting on assumptions.** With per-experiment FWHM as points (O4 a) and n = 5 per duration, the slope is negative under every ROI reading and fit range, p = 0.0008 to 0.0058 against duration and against log duration (paper p = 0.003). On the three group means (O4 b) the slopes are negative with p = 0.048 to 0.165. With all seven 600-ns wells no slope is significant (p = 0.86 to 0.99, and the slopes against log duration are positive), because of the two 600-ns wells without a central peak.

### fig6d-600-900-differ-from-300

> FWHM for 600-ns and for 900-ns packets is each significantly smaller than for 300-ns packets (Bonferroni-corrected t-tests, p = 0.028 for one and p = 0.0072 for the other).

In [ ]:
rows = []
for label, groups in (("main", f6_main), ("all 600", f6)):
    for roi in ROIS:
        for hr in RANGES:
            fw = {d: fwhm_each(g, roi, hr) for d, g in groups.items()}
            for d in ("600", "900"):
                t = stats.ttest_ind(fw[d], fw["300"])
                rows.append(dict(set=label, roi=roi, range=hr, vs300=d, diff=fw[d].mean() - fw["300"].mean(), p=t.pvalue, p_x2=min(1, 2 * t.pvalue), p_x3=min(1, 3 * t.pvalue)))
R6p = pd.DataFrame(rows); print(R6p.round(4).to_string())
mm = R6p[R6p.set == "main"]
for d in ("600", "900"):
    q = mm[mm.vs300 == d]
    print(f"main, {d} vs 300: smaller FWHM in every reading {bool((q['diff'] < 0).all())}; corrected p (x2) {q.p_x2.min():.4f} to {q.p_x2.max():.4f}; (x3) {q.p_x3.min():.4f} to {q.p_x3.max():.4f}")

**Result: partially reproduced.** 900 against 300 ns: FWHM smaller by 0.36 to 0.40 mm, corrected p 0.0063 to 0.0118 (two comparisons) and 0.0095 to 0.0178 (three) under every reading, near the paper's 0.0072. 600 against 300 ns: smaller by 0.25 to 0.28 mm, corrected p 0.049 to 0.169 (two) and 0.074 to 0.253 (three), below 0.05 under one reading only.

### fig6d-900ns-fwhm

> Packets of 900-ns pulses reduced FWHM to 1.7 +/- 0.08 mm, 11.1 +/- 0.6% of the gap between electrodes.

In [ ]:
for roi in ROIS:
    for hr in RANGES:
        v = fwhm_each(f6["900"], roi, hr)
        m, se, n = ms(v)
        print(f"ROI {roi}, range {hr}: 900-ns FWHM {m:.3f} +/- {se:.3f} mm (n={n}), {100 * m / cc.DIAG_GAP_MM:.2f} +/- {100 * se / cc.DIAG_GAP_MM:.2f}% of 15.3 mm; within 1.7 +/- 0.08: {abs(m - 1.7) <= 0.08}; within 11.1 +/- 0.6: {abs(100 * m / 15.3 - 11.1) <= 0.6}")

**Result: reproduced.** 900-ns FWHM is 1.688 to 1.767 mm (s.e. 0.049 to 0.052, n = 5), 11.03 to 11.55% of 15.3 mm, inside 1.7 +/- 0.08 mm and 11.1 +/- 0.6% under every reading.

## Figure 7: number of bursts (DANDI:001276)

0.2-MHz CANCAN wells by cell line and number of bursts (x4). The dandiset holds more CHO wells than the paper's n = 4 (printed in the data check); all are used, and the per-date breakdown is shown.

In [ ]:
B = {}
for cell in ("CT26", "CHO"):
    for b in ("1", "2", "4", "6"):
        B[(cell, b)] = wells("001276", cell=cell, repeats=b, packet_rate="0.2MHz", protocol="CanCan")
print({k: len(v) for k, v in B.items()})
print({k: sorted({o["subject"][3:11] for o in v}) for k, v in B.items()})

### fig7be-center-linear-with-bursts

> The central YP peak grew linearly with the number of bursts in both CT26 and CHO cells (slopes different from zero at p < 0.001), except that it saturated after 6 bursts (x4) in CT26 cells.

In [ ]:
F7 = {}
for cell in ("CT26", "CHO"):
    bs = ("1", "2", "4") if cell == "CT26" else ("1", "2", "4", "6")
    X = np.concatenate([[int(b)] * len(B[(cell, b)]) for b in bs]); Y = np.concatenate([[at(o, 0) for o in B[(cell, b)]] for b in bs])
    st = cc.slope_test(X, Y); F7[cell] = (X, Y, st)
    Xm = np.array([int(b) for b in bs]); Ym = np.array([np.mean([at(o, 0) for o in B[(cell, b)]]) for b in bs])
    sm = cc.slope_test(Xm, Ym)
    print(f"{cell}: means by bursts {dict(zip(bs, np.round(Ym)))}; experiments n={len(X)}: slope {st['slope']:.1f} a.u./burst, r = {st['r']:.3f}, p = {st['p']:.2e}; group means: slope {sm['slope']:.1f}, p = {sm['p']:.4f}")
    if cell == "CT26":
        X6 = np.concatenate([X, [6] * len(B[("CT26", "6")])]); Y6 = np.concatenate([Y, [at(o, 0) for o in B[("CT26", "6")]]])
        s6 = cc.slope_test(X6, Y6); print(f"   CT26 with 6 bursts kept (variant): slope {s6['slope']:.1f}, p = {s6['p']:.2e}")
    # per date
    for date in sorted({o["subject"][3:11] for b in bs for o in B[(cell, b)]}):
        xx = [int(b) for b in bs for o in B[(cell, b)] if o["subject"][3:11] == date]
        yy = [at(o, 0) for b in bs for o in B[(cell, b)] if o["subject"][3:11] == date]
        print(f"   {date}: {len(xx)} wells, bursts {sorted(set(xx))}")

**Result: reproduced, resting on an assumption.** On individual experiments (O4 a), centre YP grows with bursts in CT26 (1 to 4 bursts, n = 12, slope 903 a.u. per burst, p = 9.1e-04) and CHO (1 to 6 bursts, n = 28, slope 606, p = 2.4e-11). On group means (O4 b) the CT26 slope has p = 0.036 and the CHO slope p = 0.0015, which would make the claim partially reproduced.

### fig7b-ct26-saturation

> In CT26 cells the central YP uptake saturated at 6 bursts (x4), falling below the linear increase seen at fewer bursts, as expected when all central cells are already killed.

In [ ]:
def band_at(X, Y, x0, level=0.95):
    X = np.asarray(X, float); Y = np.asarray(Y, float); n = len(X)
    st = cc.slope_test(X, Y); pred = st["intercept"] + st["slope"] * x0
    res = Y - (st["intercept"] + st["slope"] * X); s = np.sqrt(np.sum(res ** 2) / (n - 2))
    se = s * np.sqrt(1 / n + (x0 - X.mean()) ** 2 / np.sum((X - X.mean()) ** 2))
    q = stats.t.ppf(0.5 + level / 2, n - 2)
    return pred, pred - q * se, pred + q * se
c6 = np.array([at(o, 0) for o in B[("CT26", "6")]]); c4 = np.array([at(o, 0) for o in B[("CT26", "4")]])
X, Y, _ = F7["CT26"]
pred, lo, hi = band_at(X, Y, 6)
print(f"O4 (a), experiments: line at 6 bursts {pred:.0f}, 95% band {lo:.0f} to {hi:.0f}; 6-burst mean {c6.mean():.0f} +/- {ms(c6)[1]:.0f} (n={len(c6)}); below the band: {c6.mean() < lo}")
Xm = np.array([1, 2, 4]); Ym = np.array([np.mean([at(o, 0) for o in B[("CT26", b)]]) for b in ("1", "2", "4")])
pm, lom, him = band_at(Xm, Ym, 6)
print(f"O4 (b), group means: line at 6 {pm:.0f}, 95% band {lom:.0f} to {him:.0f}; below the band: {c6.mean() < lom}")
print(f"reading (b): 6-burst {c6.mean():.0f} +/- {ms(c6)[1]:.0f} against 4-burst {c4.mean():.0f} +/- {ms(c4)[1]:.0f}; 6 above 4 by a gap larger than the error bars: {c6.mean() > c4.mean() and cc.gap_exceeds(c6.mean(), ms(c6)[1], c4.mean(), ms(c4)[1])}")
Xc, Yc, _ = F7["CHO"]
Xc5 = Xc[Xc < 6]; Yc5 = Yc[Xc < 6]; pc, loc, hic = band_at(Xc5, Yc5, 6)
cc6 = Yc[Xc == 6]
print(f"CHO (reported): line through 1-4 bursts at 6: {pc:.0f} ({loc:.0f} to {hic:.0f}); 6-burst mean {cc6.mean():.0f} +/- {ms(cc6)[1]:.0f}")

**Result: partially reproduced.** The CT26 6-burst mean (4847 +/- 725 a.u., n = 3) lies below the line through 1 to 4 bursts (5719 at 6 bursts) but inside its 95% band (4028 to 7410 on experiments; 3204 to 8225 on group means). It is not above the 4-burst mean (3869 +/- 55) by a gap larger than the error bars, so reading (b) holds and reading (a) does not. CHO: 6-burst mean 3045 against 3194 from its 1-to-4-burst line, inside that band (2526 to 3861).

### fig7cf-fwhm-with-bursts

> The FWHM of the central peak increased in proportion to the number of bursts in CT26 cells (slope p < 0.001) but not in CHO cells (p = 0.44).

Per-experiment FWHM against bursts; CHO 1-burst wells left out (Fig. 7 legend); CT26 6-burst kept, left out as a variant. Readings O1, O3 and O4.

In [ ]:
rows = []
FW7 = {}
for roi in ROIS:
    for hr in RANGES:
        r = dict(roi=roi, range=hr)
        for cell, bs in (("CT26", ("1", "2", "4", "6")), ("CT26_no6", ("1", "2", "4")), ("CHO", ("2", "4", "6"))):
            cl = cell.split("_")[0]
            fw = {b: fwhm_each(B[(cl, b)], roi, hr) for b in bs}
            FW7[(roi, hr, cell)] = fw
            X = np.concatenate([[int(b)] * len(v) for b, v in fw.items()]); Y = np.concatenate(list(fw.values()))
            ok = np.isfinite(Y) & (Y < 50)
            st = cc.slope_test(X[ok], Y[ok]); sm = cc.slope_test([int(b) for b in bs], [np.nanmean(v) for v in fw.values()])
            r.update({f"{cell}_slope": st["slope"], f"{cell}_p": st["p"], f"{cell}_p_means": sm["p"], f"{cell}_dropped": int((~ok).sum())})
            r.update({f"{cell}_fwhm{b}": np.nanmean(v) for b, v in fw.items()})
        rows.append(r)
R7c = pd.DataFrame(rows); print(R7c.T.to_string(float_format=lambda v: f"{v:.3g}"))

**Result: reproduced, resting on an assumption.** On individual experiments the CT26 FWHM slope (1 to 6 bursts) is 0.95 to 1.22 mm per burst with p below 0.001 under every reading, and the CHO slope (2 to 6 bursts) 0.08 to 0.14 mm per burst with p = 0.068 to 0.33, not significant. On group means the CT26 p is 0.0005 to 0.016 and the CHO p 0.44 to 0.64 (paper 0.44).

### fig7-sharpest-peak-one-burst

> One burst (x4) in CT26 cells gave the sharpest peak of the study, FWHM 1.1 +/- 0.1 mm, 7.3% of the inter-electrode gap and 2.3% of the electrode array area.

Area readings: (a) disk of diameter FWHM over the square with 15.3-mm diagonals (117 mm2); (b) disk of radius FWHM over the square of the electrode centres (12.8-mm sides); (c) disk of diameter FWHM over the square of the electrode centres. The comparison groups are every group fitted per experiment in 001276 and 001277 (the Fig. 4C groups are not in any dandiset).

In [ ]:
for roi in ROIS:
    for hr in RANGES:
        v = fwhm_each(B[("CT26", "1")], roi, hr); m, se, n = ms(v)
        A_a = np.pi * (m / 2) ** 2 / (cc.DIAG_GAP_MM ** 2 / 2); A_b = np.pi * m ** 2 / cc.SIDE_MM ** 2; A_c = np.pi * (m / 2) ** 2 / cc.SIDE_MM ** 2
        groups = {f"001276 {c} {b}": np.nanmean(fwhm_each(B[(c, b)], roi, hr)) for c in ("CT26", "CHO") for b in ("1", "2", "4", "6")}
        groups.update({f"001277 {d} ns": np.nanmean(fwhm_each(g, roi, hr)) for d, g in f6_main.items()})
        groups.pop("001276 CHO 1")
        smallest = min(groups, key=groups.get)
        print(f"ROI {roi}, range {hr}: CT26 1 burst FWHM {m:.3f} +/- {se:.3f} mm (n={n}), {100 * m / 15.3:.2f}% of 15.3 mm; area (a) {100 * A_a:.2f}%, (b) {100 * A_b:.2f}%, (c) {100 * A_c:.2f}%; smallest group: {smallest} ({groups[smallest]:.3f}); next: {sorted(groups.values())[1]:.3f}")
print("per well (ROI a, 4 mm):", [(o["subject"], round(cc.fit(*prof(o), "lorentz", 4.0)["fwhm"], 3)) for o in B[("CT26", "1")]])
print("CHO 1 burst (left out of the comparison, Fig. 7 legend):", np.round(fwhm_each(B[("CHO", "1")], "a", 4.0), 2).tolist())

**Result: reproduced.** CT26 after 1 burst (x4): FWHM 1.031 to 1.134 mm (s.e. 0.15 to 0.17, n = 3), 6.74 to 7.41% of 15.3 mm; as a disk of radius FWHM over the square of electrode centres (area reading b) 2.04 to 2.47% of the array, inside 2.3 +/- 0.23% under contiguous ROIs (2.41 to 2.47%). All three values are within tolerance under contiguous ROIs, and the FWHM is the smallest group mean in 001276 and 001277 under every reading (next: 900 ns, 1.69 to 1.77 mm). The Fig. 4C groups are in no dandiset.

### fig7-offtarget-40-50-fold

> In the burst-number experiments with MHz-CANCAN, YP emission near the electrodes was 40 to 50 times smaller than at the remote target in the centre.

Ratio of centre YP to YP at 6 or 7 mm (O5 a, b), as the ratio of group means (i) or the mean of per-experiment ratios (ii), under O1; groups (A) CT26 1 burst, (B) CT26 1, 2 and 4 bursts, (C) every CHO burst number. Background-subtracted values; a near-electrode value near zero makes a per-experiment ratio unstable, so (ii) is also given as the median.

In [ ]:
rows = []
for cell in ("CT26", "CHO"):
    for b in ("1", "2", "4", "6"):
        g = B[(cell, b)]
        for roi in ROIS:
            c0 = np.array([at(o, 0, roi) for o in g])
            for d in (6, 7):
                y = np.array([at(o, d, roi) for o in g])
                rows.append(dict(cell=cell, bursts=b, roi=roi, dist=d, n=len(g), ratio_of_means=c0.mean() / y.mean(), mean_of_ratios=np.mean(c0 / y), median_ratio=np.median(c0 / y), near_mean=y.mean(), near_se=ms(y)[1], centre=c0.mean()))
R7o = pd.DataFrame(rows); R7o["in_36_55_i"] = R7o.ratio_of_means.between(36, 55); R7o["in_36_55_ii"] = R7o.mean_of_ratios.between(36, 55)
print(R7o.round(2).to_string())
# background variant: the mean of the four footprints instead of the darkest one
for b in ("1", "2"):
    g = B[("CT26", b)]
    for d in (6, 7):
        c0 = np.array([o["pa"]["raw"][:, 0].mean() - o["pa"]["backgrounds"].mean() for o in g])
        i = int(np.argmin(np.abs(cc.DIST_A - d)))
        y = np.array([o["pa"]["raw"][:, i].mean() - o["pa"]["backgrounds"].mean() for o in g])
        print(f"variant, background = mean of four footprints, CT26 {b} burst, {d} mm: ratio of means {c0.mean() / y.mean():.1f}, near-electrode mean {y.mean():.1f} a.u.")
print("background used (a.u.), CT26 1 burst:", [round(o["pa"]["background"]) for o in B[("CT26", "1")]], "all four footprints:", [np.round(o["pa"]["backgrounds"]).tolist() for o in B[("CT26", "1")]])

**Result: not reproduced.** No reading puts its groups inside 36 to 55. CT26 1 burst: 13.5 to 25.0 (ratio of means) and 13.2 to 25.5 (mean of ratios). CT26 2 bursts: 11.7 to 28.4 and 15.7 to 36.7 (one value inside the range); 4 bursts: 2.8 to 8.0. CHO groups: 1.5 to 236, inside the range for none; the largest values come where the near-electrode mean is near zero. Using the mean of the four footprints as background changes these ratios (printed above).

### fig7-cho-replicates-ct26

> Remote focusing of electroporation was consistent across the two cell types; in CHO cells as in CT26 cells the YP uptake peaked at the centre of the array.

In [ ]:
rows = []
for cell in ("CHO", "CT26"):
    for b in ("1", "2", "4", "6"):
        g = B[(cell, b)]
        for roi in ROIS:
            c0 = np.array([at(o, 0, roi) for o in g]); m0, s0, n = ms(c0)
            r = dict(cell=cell, bursts=b, roi=roi, n=n, centre=m0, se=s0)
            for d in (6, 7):
                y = np.array([at(o, d, roi) for o in g]); m, s, _ = ms(y)
                r[f"y{d}"] = m; r[f"se{d}"] = s; r[f"gap{d}"] = (m0 > m) and cc.gap_exceeds(m0, s0, m, s)
            rows.append(r)
R7x = pd.DataFrame(rows); print(R7x.round(1).to_string())
q = R7x[R7x.bursts.isin(["2", "4", "6"])]
print("bursts 2 to 6, every cell line and reading, centre above 6 and 7 mm by a gap:", bool((q.gap6 & q.gap7).all()))

**Result: reproduced.** For 2, 4 and 6 bursts in both cell lines, mean centre YP exceeds mean YP at 6 and at 7 mm by more than the error bars under both ROI readings (CHO centre 578 to 3045 a.u., near the electrodes 8 to 537 a.u.). One burst also holds in both lines.

## Summary

| Claim | Outcome |
|---|---|
| fig1e-field-simulation | blocked: data_missing |
| fig2-heating-below-44c | blocked: data_missing |
| fig3a-centre-enhanced | blocked: data_missing |
| fig3bc-rate-suppresses-off-target | blocked: data_missing |
| fig3-lorentzian-profile | blocked: data_missing |
| fig3-tail-deviation-falls-with-rate | blocked: data_missing |
| fig4b-40-fold-ratio | blocked: data_missing |
| fig4b-centre-no-rate-effect | blocked: data_missing |
| fig4b-near-electrode-decline | blocked: data_missing |
| fig4c-fwhm-falls-with-rate | blocked: data_missing |
| fig4c-log-trend | blocked: data_missing |
| fig5-raised-electrodes-weaken-centre | partially reproduced |
| fig5-killing-focused-centre | reproduced |
| fig5ab-yp-spot-slightly-larger-than-pr | not reproduced |
| fig5b-gaussian-pr-lorentzian-yp | reproduced |
| fig5b-reversible-beyond-2mm | reproduced |
| fig5c-nuclear-yp-centre | partially reproduced |
| fig6b-peak-900ns-not-significant | reproduced |
| fig6c-normalized-decline-similar | not reproduced |
| fig6d-longer-pulses-sharper-peak | reproduced (assumed) |
| fig6d-600-900-differ-from-300 | partially reproduced |
| fig6d-900ns-fwhm | reproduced |
| fig7be-center-linear-with-bursts | reproduced (assumed) |
| fig7b-ct26-saturation | partially reproduced |
| fig7cf-fwhm-with-bursts | reproduced (assumed) |
| fig7-sharpest-peak-one-burst | reproduced |
| fig7-offtarget-40-50-fold | not reproduced |
| fig7-cho-replicates-ct26 | reproduced |

A failure to reproduce here may be a failure of this attempt, not of the paper: see the reasoning and the differences from the paper's method.

How the analysis differs from the paper's:

- The ROIs are placed by code on images reduced to 26-um blocks, at whole-mm distances or in contiguous steps (the paper does not give its positions), along diagonals found from the electrode footprints. Where the footprints could not be found (most 1- and 2-mm wells, and wells with faint footprints), the centre is the centre of the central YP spot, with a fixed rotation; this is an assumption. In wells where both are known the two centres are 0.36 mm apart (median).
- Background is the mean of 3 ROIs in the darkest of the four footprints.
- The dandisets hold more wells than the paper reports for several groups (001284 YP, 001276 CHO); all are used. In 001277 two 600-ns wells without a central peak are left out so that n = 5 as in the paper; this is an assumption, and every claim on Figure 6 is also shown with them.
- Per-experiment fits are used as the points of the regressions of Figures 6D and 7 (claims file O4, reading a); the group-mean reading is shown.

What was found about the datasets:

- DANDI:001047 holds images from a 3.8-mm array with a 3.2-kV first pulse, not the paper's 15.3-mm array; the packet-rate series of Figures 3 and 4 is in none of the four dandisets. No dandiset holds the field simulation, the thermochromic film images, or the high-resolution images of Figure 5C.
- 001276 holds CANCAN wells at 1 Hz and control wells that its description does not mention, and more CHO wells than the paper's n = 4.
- 001277: per-file metadata give 9 packets at 0.2 MHz for every pulse duration; two 600-ns wells (P1-20240926-B2, B3) show no central peak; one well is a sham exposure; the Cy3-DAPI image of well P2-20240925-B3 is deposited twice.
- 001284: the FITC image of well P1-20240828-A3 is almost empty (11 MB, 98% zero pixels); YP groups hold 12, 11 and 11 wells against the paper's 6 to 7; two Hoechst files state 900-ns pulses in wells whose other files state 600 ns.
- The files give a 10-mm inter-electrode distance, which is none of the distances the paper gives (15.3-mm diagonal between needle edges, 12.8 mm between centres along a side).

In [ ]:
print(f"total run time {(time.time() - T0) / 60:.1f} min; peak memory {resource.getrusage(resource.RUSAGE_SELF).ru_maxrss / 1e6:.2f} GB")